# Batch & Stream Ingestion in Practice

Differences between batch and streaming processing. Practical ingestion patterns using COPY INTO, Auto Loader, and structured streaming.

| Training Block | Duration | Type |
|---|---|---|
| Batch & Stream Ingestion — Demo | 40 min | Demo |
| Batch & Stream Ingestion — Workshop | 30 min | Hands-on |

**Prerequisites:** Databricks Fundamentals (familiar with `spark.read`, Delta tables, basic DataFrame operations)

## Learning Objectives

After completing this module you will be able to:

- **Compare** batch vs streaming processing approaches and when to use each
- **Apply** `COPY INTO` for batch ingestion to Delta tables (idempotent, tracked)
- **Configure** Auto Loader with `cloudFiles` for scalable file ingestion
- **Use** `trigger(availableNow=True)` for batch-oriented Auto Loader scenarios
- **Filter** files with `pathGlobFilter` for multi-format landing zones
- **Handle** schema evolution and rescued data in streaming pipelines
- **Monitor** streaming queries with checkpoints and metrics

## Setup

Initialize the environment, import libraries, and prepare simulated data sources for all ingestion demos in this module.

---

In [0]:
%run ../../setup/00_setup

### Configuration

Import libraries and define paths for source data, checkpoints, and schema locations.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import time

In [0]:
# Set default catalog and schema
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

# === SOURCE DATA (REAL DATASET) ===
SOURCE_CUSTOMERS = f"{DATASET_PATH}/customers/customers.csv"
SOURCE_ORDERS = f"{DATASET_PATH}/orders/orders_batch.json"

# === DEMO PATHS (SIMULATED ARRIVAL) ===
DEMO_BASE_PATH = f"{DATASET_PATH}/ingestion_demo"
BATCH_SOURCE_PATH = f"{DEMO_BASE_PATH}/batch_source"
STREAM_SOURCE_PATH = f"{DEMO_BASE_PATH}/stream_source"

# === TECHNICAL PATHS ===
CHECKPOINT_BASE_PATH = f"{DEMO_BASE_PATH}/checkpoints"
SCHEMA_BASE_PATH = f"{DEMO_BASE_PATH}/schemas"
BAD_RECORDS_PATH = f"{DEMO_BASE_PATH}/bad_records"

# Cleanup from previous runs
dbutils.fs.rm(DEMO_BASE_PATH, True)
print(f"Demo environment prepared at: {DEMO_BASE_PATH}")

In [0]:
display(
    spark.createDataFrame([
        ("CATALOG", CATALOG),
        ("BRONZE_SCHEMA", BRONZE_SCHEMA),
        ("SILVER_SCHEMA", SILVER_SCHEMA),
        ("USER", raw_user),
        ("CUSTOMERS_CSV", SOURCE_CUSTOMERS),
        ("STREAMING_SOURCE_PATH", STREAM_SOURCE_PATH)
    ], ["Variable", "Value"])
)

In [0]:
# === DATA PREPARATION (SIMULATION) ===

# 1. Prepare Batch Data (Customers)
# We split customers into 2 days for COPY INTO demo
df_customers = spark.read.option("header", "true").csv(SOURCE_CUSTOMERS)
df_batch_day1, df_batch_day2,df_batch_day3,df_batch_day4 = df_customers.randomSplit([0.25]*4, seed=42)

# Save Day 1 immediately
df_batch_day1.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day1")
print(f"Batch Data: Day 1 ready at {BATCH_SOURCE_PATH}/day1")

# 2. Prepare Streaming Data (Orders)
# We take existing stream files, merge them, and split into 10 micro-batches for simulation
SOURCE_STREAM_FILES = f"{DATASET_PATH}/orders/stream/*.json"
df_all_orders = spark.read.json(SOURCE_STREAM_FILES)

# Split into 20 parts (5% each)
stream_batches = df_all_orders.randomSplit([0.05] * 20, seed=42)

# Save Batch 1 immediately to start the stream
stream_batches[0].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_01")
print(f"Stream Data: Batch 1 ready at {STREAM_SOURCE_PATH}/batch_01")

### Data Loading Methods Overview

<img src="../../../assets/images/d271b8dfc29049aaab22d97536aca66d.avif" width="800">

| Feature | CTAS | COPY INTO | Auto Loader |
|---------|------|-----------|-------------|
| **Incremental** | No | Yes | Yes |
| **Idempotent** | No | Yes | Yes |
| **Schema Evolution** | No | Limited | Advanced |
| **File Tracking** | No | Metadata | Checkpoint |
| **Scalability** | Low | Medium | High |
| **Streaming** | No | No | Yes |
| **Use Case** | One-time | Scheduled batch | Real-time/Streaming |

> **Pro Tip:** Auto Loader (`cloudFiles`) is the **recommended** ingestion method for new projects. COPY INTO is suitable for scheduled batch loads with <100K files.

---

### CTAS — Create Table As Select

The simplest way to create a Delta table from a query result. CTAS reads data once and writes a snapshot — no tracking, no incremental processing.

```sql
CREATE OR REPLACE TABLE target AS
SELECT ... FROM source
```

| Feature | CTAS |
|---------|------|
| **Incremental** | No — full overwrite every time |
| **Idempotent** | No — `CREATE OR REPLACE` drops & recreates |
| **Schema evolution** | No — fixed at creation |
| **File tracking** | No |
| **Best for** | One-time loads, prototyping, small tables |

> **Key limitation:** CTAS does NOT track which files were already loaded. Every re-run overwrites the entire table. For incremental loading, use **COPY INTO** or **Auto Loader**.

In [0]:
# Example: Load CSV from volume to customer_cts table using CTAS

table_name = "customer_cts"

display(spark.sql(f"""
CREATE OR REPLACE TABLE {table_name} AS
SELECT *
FROM csv.`{SOURCE_CUSTOMERS}`
"""))

In [0]:
%sql
select * from customer_cts

**CTAS with transformations — type casting, filtering, computed columns:**

In [0]:
# CTAS with transformations — real-world pattern

display(spark.sql(f"""
CREATE OR REPLACE TABLE customer_cts_enriched AS
SELECT
  customer_id,
  UPPER(first_name) AS first_name,
  UPPER(last_name) AS last_name,
  LOWER(email) AS email,
  city,
  state,
  country,
  TO_DATE(registration_date, 'yyyy-MM-dd') AS registration_date,
  customer_segment,
  DATEDIFF(current_date(), TO_DATE(registration_date, 'yyyy-MM-dd')) AS days_since_registration,
  current_timestamp() AS _ingestion_timestamp
FROM read_files('{SOURCE_CUSTOMERS}', format => 'csv', header => true)
WHERE customer_segment IS NOT NULL
"""))

display(spark.table("customer_cts_enriched"))

**CTAS is NOT incremental — re-run overwrites everything:**

In [0]:
# Demonstrate: CTAS always overwrites
count_before = spark.table("customer_cts").count()

# Re-run the SAME CTAS — it drops & recreates the table
spark.sql(f"""
CREATE OR REPLACE TABLE customer_cts AS
SELECT * FROM csv.`{SOURCE_CUSTOMERS}`
""")

count_after = spark.table("customer_cts").count()

display(spark.createDataFrame([
    ("Before CTAS re-run", count_before),
    ("After CTAS re-run", count_after),
    ("Difference", count_after - count_before),
    ("Conclusion", 0),  # same count — full overwrite, not append
], ["State", "Count"]))

# Unlike COPY INTO or Auto Loader, CTAS doesn't track files.
# It always processes ALL source files from scratch.
print("CTAS = full overwrite. No file tracking. No incremental.")

**CTAS from different formats — Parquet, JSON, Delta:**

```sql
-- From Parquet
CREATE TABLE bronze_parquet AS SELECT * FROM parquet.`/path/to/files/`;

-- From JSON
CREATE TABLE bronze_json AS SELECT * FROM json.`/path/to/files/`;

-- From existing Delta table (snapshot copy)
CREATE TABLE gold_snapshot AS SELECT * FROM silver_table WHERE date = current_date();

-- Read from Volume path
CREATE TABLE bronze_volume AS SELECT * FROM read_files('/Volumes/catalog/schema/volume/data/');
```

> **Pro Tip:** Use `read_files()` for Volume-based data — it's the recommended approach in Unity Catalog environments. Direct format readers (`csv.`, `json.`) work but `read_files()` adds metadata columns automatically.

## COPY INTO — Batch Loading

COPY INTO provides idempotent, file-level batch ingestion from cloud storage into Delta tables, automatically tracking which files have already been loaded.

**Syntax:**
```sql
COPY INTO target_table
FROM (SELECT ... FROM 'path/')
FILEFORMAT = { CSV | JSON | PARQUET | AVRO | ORC | TEXT }
FORMAT_OPTIONS ('key' = 'value', ...)
COPY_OPTIONS ('key' = 'value', ...)
```

| Clause | Key Options | Description |
|---|---|---|
| `FILEFORMAT` | `CSV`, `JSON`, `PARQUET`, `AVRO` | Source file format |
| `FORMAT_OPTIONS` | `header`, `inferSchema`, `delimiter` | Format-specific parsing options |
| `COPY_OPTIONS` | `mergeSchema`, `force` | `mergeSchema=true` enables schema evolution; `force=true` reloads already-processed files |

> **Pro Tip:** `COPY INTO` is **idempotent** — it tracks processed files and won't load duplicates on re-run.

---

**COPY INTO — Syntax Reference**

| Element | Syntax |
|---------|--------|
| Basic load | `COPY INTO table FROM '/path/' FILEFORMAT = CSV` |
| With transform | `COPY INTO table FROM (SELECT col, CAST(ts AS TIMESTAMP) FROM '/path/') FILEFORMAT = JSON` |
| Format options | `FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')` |
| Copy options | `COPY_OPTIONS ('mergeSchema' = 'true')` |
| Force re-load | `COPY INTO table FROM '/path/' FORCE = TRUE` |
| Supported formats | `CSV` \| `JSON` \| `PARQUET` \| `AVRO` \| `ORC` \| `DELTA` |


### Demo: COPY INTO from CSV

Załadujemy plik CSV z danymi klientów do tabeli Delta za pomocą `COPY INTO`. Pokażemy jak zdefiniować tabelę docelową, dodać transformacje (casting, computed columns), i uruchomić idempotentne ładowanie.

In [0]:
TABLE_CUSTOMERS = f"{BRONZE_SCHEMA}.customers_batch"

**Creating target table:**

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE_CUSTOMERS}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_CUSTOMERS} (
  customer_id STRING,
  first_name STRING,
  last_name STRING,
  email STRING,
  phone STRING,
  city STRING,
  state STRING,
  country STRING,
  registration_date DATE,
  customer_segment STRING,
  _ingestion_timestamp TIMESTAMP
) USING DELTA
COMMENT 'Customers data - Bronze layer'
""")

**Execute COPY INTO:**

In [0]:
# Load Day 1 data
result = spark.sql(f"""
COPY INTO {TABLE_CUSTOMERS}
FROM (
  SELECT 
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    TO_DATE(registration_date, 'yyyy-MM-dd') as registration_date,
    customer_segment,
    current_timestamp() as _ingestion_timestamp
  FROM '{BATCH_SOURCE_PATH}/day1'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

display(result)
display(spark.table(TABLE_CUSTOMERS))

### Demo: Idempotency

Uruchomimy `COPY INTO` ponownie na tych samych danych i zobaczymy, że żadne duplikaty nie zostały dodane — mechanizm file tracking automatycznie pomija już załadowane pliki.

In [0]:
count_before = spark.table(TABLE_CUSTOMERS).count()

# Re-run COPY INTO (same source path)
spark.sql(f"""
COPY INTO {TABLE_CUSTOMERS}
FROM (
  SELECT 
    customer_id, first_name, last_name, email, phone,
    city, state, country,
    TO_DATE(registration_date, 'yyyy-MM-dd') as registration_date,
    customer_segment,
    current_timestamp() as _ingestion_timestamp
  FROM '{BATCH_SOURCE_PATH}/*'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
""")

In [0]:
count_after = spark.table(TABLE_CUSTOMERS).count()
display(count_after)

In [0]:
display(
    spark.createDataFrame([
        ("Before", count_before),
        ("After", count_after),
        ("Difference", count_after - count_before)
    ], ["State", "Count"])
)

### Demo: Adding More Days

Zasymulujemy dostarczanie danych w kolejnych dniach (day2, day3, day4). Po dodaniu nowych plików, `COPY INTO` automatycznie załaduje tylko nowe pliki — bez ponownego przetwarzania day1.

In [0]:
# Add Day 2 data
count_before_days = spark.table(TABLE_CUSTOMERS).count()
print(f"Current row count: {count_before_days}")

df_batch_day2.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day2")
print(f"Day 2 data written to {BATCH_SOURCE_PATH}/day2")

In [0]:
# Add Day 3 data
df_batch_day3.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day3")
print(f"Day 3 data written to {BATCH_SOURCE_PATH}/day3")

In [0]:
# Add Day 4 data
df_batch_day4.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day4")
print(f"Day 4 data written to {BATCH_SOURCE_PATH}/day4")

In [0]:
# Now COPY INTO picks up only the NEW files (day2, day3, day4)
result = spark.sql(f"""
COPY INTO {TABLE_CUSTOMERS}
FROM (
  SELECT 
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    TO_DATE(registration_date, 'yyyy-MM-dd') as registration_date,
    customer_segment,
    current_timestamp() as _ingestion_timestamp
  FROM '{BATCH_SOURCE_PATH}/*'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
""")

count_after_days = spark.table(TABLE_CUSTOMERS).count()
print(f"Count before: {count_before_days}")
print(f"Count after:  {count_after_days}")
print(f"New rows:     {count_after_days - count_before_days}")

display(result)

## Auto Loader — Streaming Ingestion

Auto Loader (`cloudFiles`) provides scalable, checkpoint-based streaming ingestion from cloud storage with built-in schema evolution and exactly-once guarantees.

**readStream pattern:**
```python
spark.readStream
 .format("cloudFiles")
 .option("cloudFiles.format", "json|csv|parquet|...")
 .option("cloudFiles.schemaLocation", "/path/to/schema")
 .option("cloudFiles.schemaEvolutionMode", "addNewColumns|rescue|failOnNewColumns|none")
 .load("/source/path")
```

| Option | Values | Description |
|---|---|---|
| `cloudFiles.format` | `json`, `csv`, `parquet`, `avro` | Source file format |
| `cloudFiles.schemaLocation` | path | Where Auto Loader persists the inferred schema |
| `cloudFiles.schemaEvolutionMode` | `addNewColumns` (default), `rescue`, `failOnNewColumns`, `none` | How to handle schema changes |
| `cloudFiles.inferColumnTypes` | `true` / `false` | Infer precise column types (default: all strings) |
| `cloudFiles.includeExistingFiles` | `true` / `false` | Process files that existed before the stream started |
| `cloudFiles.maxFilesPerTrigger` | integer | Limits files processed per micro-batch |

> **Pro Tip:** Auto Loader uses `cloudFiles` format with **checkpoint-based** exactly-once processing. It's the **recommended** method for new ingestion pipelines.

---

### Trigger Modes & Output Modes

| Trigger Mode | Behavior | Use Case |
|------|----------|----------|
| `availableNow=True` | Process all available → stop | Scheduled jobs |
| `processingTime="10 seconds"` | Every N seconds | Real-time |
| `once=True` | Legacy (deprecated) | — |

<img src="../../../assets/images/113d4c6273584dc6aa6882e2afe85d0b.png" width="800">

| Output Mode | Description | Use Case |
|------|-------------|----------|
| **Append** | Only new rows | Raw data ingestion (stateless) |
| **Update** | Only updated rows | Aggregations (stateful) |
| **Complete** | Entire result rewritten | Small aggregations |

---

In [0]:
try:
    dbutils.fs.rm(CHECKPOINT_BASE_PATH, True)
    dbutils.fs.rm(SCHEMA_BASE_PATH, True)
    dbutils.fs.rm(BAD_RECORDS_PATH, True)
except:
    pass

In [0]:
TARGET_TABLE_AL = f"{BRONZE_SCHEMA}.orders_autoloader"
CHECKPOINT_AL = f"{CHECKPOINT_BASE_PATH}/autoloader"
SCHEMA_AL = f"{SCHEMA_BASE_PATH}/autoloader"

**Auto Loader readStream configuration:**

### Auto Loader Configuration Options

| Category | Key Options |
|---|---|
| **Common** | `cloudFiles.format`, `cloudFiles.schemaLocation`, `cloudFiles.includeExistingFiles` |
| **Schema** | `cloudFiles.inferColumnTypes`, `cloudFiles.schemaEvolutionMode` (`addNewColumns`, `rescue`, `failOnNewColumns`) |
| **File Discovery** | `cloudFiles.useIncrementalListing`, `cloudFiles.maxFilesPerTrigger` |
| **Notification** | `cloudFiles.useNotifications` (cloud-specific: SQS, Event Grid) |

[Full options reference](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/options/)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE_AL}")

In [0]:
df_autoloader = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_AL)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(STREAM_SOURCE_PATH) # Reading from our simulated stream source
)

**Adding metadata columns:**

In [0]:
from pyspark.sql.functions import col

df_enriched = (df_autoloader
    .withColumn("_processing_time", F.current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

**Start streaming with `availableNow` trigger:**

> `availableNow` - processes all available data and stops (batch-like streaming)

In [0]:
query = (df_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_AL)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE_AL)
)
display(spark.table(f"{TARGET_TABLE_AL}"))

**Auto Loader results:**

In [0]:
display(
    spark.createDataFrame([
        ("Records Loaded", str(spark.table(TARGET_TABLE_AL).count())),
        ("Source Files", str(spark.table(TARGET_TABLE_AL).select("_source_file").distinct().count()))
    ], ["Metric", "Value"])
)

### Demo: Incremental Processing — Add New Data

Dodamy nowy plik (batch_02) do katalogu źródłowego i ponownie uruchomimy Auto Loader z tym samym checkpointem. Zobaczymy, że przetworzone zostaną tylko nowe pliki.

In [0]:
# Add Batch 2 to the source (safe to rerun — idempotent write)
stream_batches[1].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_02")
print(f"Batch 02 written to {STREAM_SOURCE_PATH}/batch_02")

count_before_incr = spark.table(TARGET_TABLE_AL).count()

# Re-run with the SAME checkpoint — only new files are processed
df_incr = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_AL)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(STREAM_SOURCE_PATH)
    .withColumn("_processing_time", F.current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

query_incr = (df_incr.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_AL)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE_AL)
)
query_incr.awaitTermination(60)  # 60s timeout — availableNow=True finishes fast

count_after_incr = spark.table(TARGET_TABLE_AL).count()
print(f"Before: {count_before_incr} → After: {count_after_incr} (new rows: {count_after_incr - count_before_incr})")

### Demo: cleanSource — Automatic File Archival

In production, you want to **clean up** the landing zone after files are processed. Auto Loader supports automatic file archival via `cloudFiles.cleanSource`:

| Mode | Behavior | Use Case |
|------|----------|----------|
| `OFF` (default) | No cleanup | Development, shared sources |
| `DELETE` | Deletes files after retention period | Simple cleanup, no audit trail |
| `MOVE` | Moves files to archive path | **Recommended** — audit trail + clean landing zone |

**Key options (DBR 16.4+):**

| Option | Description | Default |
|--------|-------------|---------|
| `cloudFiles.cleanSource` | `OFF`, `DELETE`, or `MOVE` | `OFF` |
| `cloudFiles.cleanSource.moveDestination` | Archive path (required for `MOVE`) | None |
| `cloudFiles.cleanSource.retentionDuration` | Wait time before cleanup | `30 days` |

> **Pro Tip:** Use `MOVE` mode with a Volume path like `/Volumes/catalog/schema/volume/archive/` for audit compliance. The archive path must NOT be a child of the source directory (to avoid re-ingestion).

---

In [0]:
# === Demo: cleanSource with MOVE mode ===
# Auto Loader will move processed files to an archive directory

ARCHIVE_SOURCE = f"{DEMO_BASE_PATH}/archive_demo/landing"
ARCHIVE_DEST = f"{DEMO_BASE_PATH}/archive_demo/archive"
ARCHIVE_CHECKPOINT = f"{CHECKPOINT_BASE_PATH}/archive_demo"
ARCHIVE_SCHEMA = f"{SCHEMA_BASE_PATH}/archive_demo"
ARCHIVE_TABLE = f"{BRONZE_SCHEMA}.orders_archive_demo"

# Cleanup
spark.sql(f"DROP TABLE IF EXISTS {ARCHIVE_TABLE}")
dbutils.fs.rm(f"{DEMO_BASE_PATH}/archive_demo", True)

# Prepare test data in landing zone
stream_batches[3].write.mode("overwrite").json(f"{ARCHIVE_SOURCE}/batch_01")
stream_batches[4].write.mode("overwrite").json(f"{ARCHIVE_SOURCE}/batch_02")

print(f"Landing zone: {ARCHIVE_SOURCE}")
print(f"Archive path: {ARCHIVE_DEST}")
print(f"Files in landing: {[f.name for f in dbutils.fs.ls(ARCHIVE_SOURCE)]}")

In [0]:
# Auto Loader with cleanSource = MOVE
# Files will be moved to archive path after processing + retention period

df_archive = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", ARCHIVE_SCHEMA)
    .option("cloudFiles.inferColumnTypes", "true")
    # === Archive options (DBR 16.4+) ===
    .option("cloudFiles.cleanSource", "MOVE")                       # MOVE processed files
    .option("cloudFiles.cleanSource.moveDestination", ARCHIVE_DEST) # Archive location
    .option("cloudFiles.cleanSource.retentionDuration", "0 hours")  # Immediate for demo (production: 7+ days)
    .load(ARCHIVE_SOURCE)
    .withColumn("_processing_time", F.current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

query_archive = (df_archive.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", ARCHIVE_CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(ARCHIVE_TABLE)
)
query_archive.awaitTermination()

print(f"Rows loaded: {spark.table(ARCHIVE_TABLE).count()}")

In [0]:
# Verify: files moved from landing to archive
import time
time.sleep(5)  # Small delay for file move operation

try:
    landing_files = [f.name for f in dbutils.fs.ls(ARCHIVE_SOURCE)]
except:
    landing_files = ["(empty — all files moved!)"]

try:
    archive_files = [f.name for f in dbutils.fs.ls(ARCHIVE_DEST)]
except:
    archive_files = ["(no archive yet — check retentionDuration)"]

display(spark.createDataFrame([
    ("Landing zone (source)", str(landing_files)),
    ("Archive (destination)", str(archive_files)),
    ("Rows in table", str(spark.table(ARCHIVE_TABLE).count()))
], ["Location", "Contents"]))

# After processing, files are MOVED from landing → archive
# Landing zone stays clean, archive provides audit trail

### Demo: pathGlobFilter — Filtering Files by Pattern

When multiple file types or patterns exist in the same directory, use `pathGlobFilter` to process only matching files. This is essential for **multi-format pipelines** where different systems drop files to the same landing zone.

| Option | Description | Example |
|--------|-------------|----------|
| `pathGlobFilter` | Unix-style glob pattern | `*.json`, `orders_*.csv`, `2024-*/*.parquet` |

> **Use Case:** Landing zone receives `customers_*.csv`, `orders_*.json`, and `products_*.parquet`. Create 3 separate streams, each with its own `pathGlobFilter` and checkpoint.

---

In [0]:
# === Demo: pathGlobFilter for selective file processing ===
# Scenario: Process only JSON files from a mixed-format directory

GLOB_SOURCE = f"{DEMO_BASE_PATH}/glob_demo/landing"
GLOB_CHECKPOINT_JSON = f"{CHECKPOINT_BASE_PATH}/glob_json"
GLOB_CHECKPOINT_CSV = f"{CHECKPOINT_BASE_PATH}/glob_csv"
GLOB_SCHEMA = f"{SCHEMA_BASE_PATH}/glob_demo"
GLOB_TABLE_JSON = f"{BRONZE_SCHEMA}.orders_json_only"
GLOB_TABLE_CSV = f"{BRONZE_SCHEMA}.customers_csv_only"

# Cleanup
spark.sql(f"DROP TABLE IF EXISTS {GLOB_TABLE_JSON}")
spark.sql(f"DROP TABLE IF EXISTS {GLOB_TABLE_CSV}")
dbutils.fs.rm(f"{DEMO_BASE_PATH}/glob_demo", True)
dbutils.fs.rm(GLOB_CHECKPOINT_JSON, True)
dbutils.fs.rm(GLOB_CHECKPOINT_CSV, True)

# Create mixed-format landing zone
stream_batches[5].write.mode("overwrite").json(f"{GLOB_SOURCE}/orders_batch1.json")
stream_batches[6].write.mode("overwrite").json(f"{GLOB_SOURCE}/orders_batch2.json")
df_batch_day1.write.mode("overwrite").option("header", "true").csv(f"{GLOB_SOURCE}/customers_day1.csv")
df_batch_day2.write.mode("overwrite").option("header", "true").csv(f"{GLOB_SOURCE}/customers_day2.csv")

print(f"Mixed landing zone: {GLOB_SOURCE}")
print(f"Files: {[f.name for f in dbutils.fs.ls(GLOB_SOURCE)]}")

In [0]:
# Stream 1: Process ONLY JSON files (orders)
df_json_only = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{GLOB_SCHEMA}/json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("pathGlobFilter", "*.json")  # <-- Only JSON files!
    .load(GLOB_SOURCE)
    .withColumn("_source_file", col("_metadata.file_path"))
)

query_json = (df_json_only.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", GLOB_CHECKPOINT_JSON)
    .trigger(availableNow=True)
    .toTable(GLOB_TABLE_JSON)
)
query_json.awaitTermination()

print(f"JSON stream: {spark.table(GLOB_TABLE_JSON).count()} rows")
print(f"Source files: {spark.table(GLOB_TABLE_JSON).select('_source_file').distinct().collect()}")

In [0]:
# Stream 2: Process ONLY CSV files (customers)
df_csv_only = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{GLOB_SCHEMA}/csv")
    .option("header", "true")
    .option("pathGlobFilter", "*.csv")  # <-- Only CSV files!
    .load(GLOB_SOURCE)
    .withColumn("_source_file", col("_metadata.file_path"))
)

query_csv = (df_csv_only.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", GLOB_CHECKPOINT_CSV)
    .trigger(availableNow=True)
    .toTable(GLOB_TABLE_CSV)
)
query_csv.awaitTermination()

print(f"CSV stream: {spark.table(GLOB_TABLE_CSV).count()} rows")
print(f"Source files: {spark.table(GLOB_TABLE_CSV).select('_source_file').distinct().collect()}")

In [0]:
# Verify: Each stream processed only matching files
display(spark.createDataFrame([
    ("JSON stream (orders)", spark.table(GLOB_TABLE_JSON).count(), 
     spark.table(GLOB_TABLE_JSON).select("_source_file").distinct().count()),
    ("CSV stream (customers)", spark.table(GLOB_TABLE_CSV).count(),
     spark.table(GLOB_TABLE_CSV).select("_source_file").distinct().count()),
], ["Stream", "Rows", "Source Files"]))

# Key insight: Same landing zone, different streams, no file conflicts!

### Demo: Continuous Processing (processingTime)

Uruchomimy strumień w trybie ciągłym z `processingTime="10 seconds"`, a następnie będziemy symulować przybycie nowych batchów co kilka sekund. Zobaczymy jak stream automatycznie podnosi nowe pliki w kolejnych micro-batchach.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE_AL}_continuous")

In [0]:
# Start stream in background
query_continuous = (df_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_AL}_continuous")
    .trigger(processingTime="10 seconds")
    .toTable(f"{TARGET_TABLE_AL}_continuous")
)

import time
print("Stream started... Waiting for first micro-batch...")
time.sleep(15)  # Wait for first micro-batch to complete
display(spark.table(f"{TARGET_TABLE_AL}_continuous").count())

In [0]:
# Simulate arrival of batches 3 to 10
print("Starting data simulation...")

for i in range(2, 10): # stream_batches indices 2-9 → batch_03 to batch_10
    batch_num = i + 1
    print(f"Arriving: Batch {batch_num}...", end=" ")
    
    # Write next batch
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_{batch_num:02d}")
    
    print("Done. Waiting for stream...")
    time.sleep(4) # Wait for trigger to pick it up

print("All batches arrived.")
display(spark.table(f"{TARGET_TABLE_AL}_continuous").count())
display(spark.table(f"{TARGET_TABLE_AL}_continuous"))

In [0]:
# Stop stream
query_continuous.stop()

## Stream-Static Joins & Aggregations

> **SKIP IF SHORT ON TIME** — This section demonstrates stream-static joins and aggregation patterns. Can be deferred to workshop.


Stream-Static Join = enriching a stream (e.g., Orders) with a static table (e.g., Customers). Static table is re-read at each micro-batch start.

---

In [0]:
# Load Static Table
df_static_customers = spark.table(TABLE_CUSTOMERS)
display(df_static_customers)

### Demo: Stream-Static Join (Append Mode)

Połączymy strumień zamówień (orders) z tabelą statyczną klientów (customers), aby wzbogacić dane o imię, nazwisko i email. Wynik zapiszemy w trybie append.

In [0]:
spark.sql("DROP TABLE IF EXISTS joined_orders")

In [0]:
df_enriched.createOrReplaceTempView("enriched_orders_stream")
df_static_customers.createOrReplaceTempView("static_customers")

df_joined = spark.sql("""
SELECT
  o.order_id,
  o.total_amount,
  c.first_name,
  c.last_name,
  c.email,
  o._processing_time
FROM enriched_orders_stream o
LEFT JOIN static_customers c
  ON o.customer_id = c.customer_id
""")

In [0]:
# Write enriched stream
query_join = (df_joined.writeStream
    .format("memory")
    .queryName("enriched_orders")
    .outputMode("append") # Joins (Left Stream-Static) are append-only
    .option("checkpointLocation", f"{CHECKPOINT_BASE_PATH}/join_demo_3")
    .trigger(processingTime="5 seconds")
    .start()
)

In [0]:
# Simulate arrival of batches 12-13 for join demo
print("Starting data simulation...")

for i in range(11, 13): # stream_batches indices 11-12 → batch_12 to batch_13
    batch_num = i + 1
    print(f"Arriving: Batch {batch_num}...", end=" ")
    
    # Write next batch
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_{batch_num:02d}")
    
    print("Done. Waiting for stream...")
    time.sleep(4) # Wait for trigger to pick it up

print("All batches arrived.")
display(spark.table("enriched_orders"))

In [0]:
query_join.stop()
display(spark.sql("SELECT count(1) FROM enriched_orders "))

### Demo: Streaming Aggregation (Watermarking)

Zdefiniujemy agregację z oknem czasowym (30 sekund) i watermarkiem (1 minuta tolerancji na opóźnione dane). Strumień w trybie `update` emituje zmieniające się wyniki na bieżąco.

In [0]:
# 1. Define Streaming Aggregation (Orders per 30 seconds)
# We use the previously defined df_enriched (from Auto Loader)

windowed_counts = (df_enriched
    .withWatermark("_processing_time", "1 minutes") # Allow 1 mins late data
    .groupBy(
        F.window("_processing_time", "30 seconds"),
        "customer_id"
    )
    .count()
)

# 2. Write Stream with UPDATE mode
# Update mode is efficient for aggregations - it emits only changed windows
query_agg = (windowed_counts.writeStream
    .format("console") 
    .queryName("orders_counts")
    .outputMode("update")
    .option("checkpointLocation", f"{CHECKPOINT_BASE_PATH}/agg_demo")
    .trigger(processingTime="5 seconds")
    .start()
)

print("Aggregation stream started...")

In [0]:
# Simulate arrival of batches 15-16 for aggregation demo
print("Starting data simulation...")

for i in range(14, 16): # stream_batches indices 14-15 → batch_15 to batch_16
    batch_num = i + 1
    print(f"Arriving: Batch {batch_num}...", end=" ")
    
    # Write next batch
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_{batch_num:02d}")
    
    print("Done. Waiting for stream...")
    time.sleep(4) # Wait for trigger to pick it up

print("All batches arrived.")
display(windowed_counts)

In [0]:
# Stop for demo purposes
query_agg.stop()

## Error Handling

> **SKIP IF SHORT ON TIME** — Error handling modes can be covered briefly (show the table only) and practiced in the workshop.


Databricks provides multiple strategies for handling malformed, corrupted, or schema-mismatched records during ingestion.

| Mode | Behavior |
|------|----------|
| `PERMISSIVE` | Parses what it can, errors → `_corrupt_record` |
| `DROPMALFORMED` | Removes malformed records |
| `FAILFAST` | Stops on first error |

> **Best Practice:** Use `badRecordsPath` to save malformed records for later analysis.

---

### Schema Evolution & Rescued Data

| `schemaEvolutionMode` | Behavior |
|---|---|
| `addNewColumns` | Automatically adds new columns |
| `rescue` | New/mismatched → `_rescued_data` JSON column |
| `failOnNewColumns` | Fail if schema changes |
| `none` | Ignores new columns |

> **Pro Tip:** `_rescued_data` column captures new columns, type mismatches, and malformed records when using `rescue` mode.

---

In [0]:
TARGET_TABLE_RESCUE = f"{BRONZE_SCHEMA}.orders_rescued"
CHECKPOINT_RESCUE = f"{CHECKPOINT_BASE_PATH}/rescue"
SCHEMA_RESCUE = f"{SCHEMA_BASE_PATH}/rescue"

spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE_RESCUE}")

**Define explicit schema (partial):**

In [0]:
# Deliberately define only some columns - rest will go to _rescued_data
partial_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("total_amount", DoubleType(), True)
])

**Auto Loader with rescue mode:**

In [0]:
# Create BAD data (Extra column + Type mismatch)
bad_data = [{"order_id": 99999, "total_amount": "INVALID_NUMBER", "new_col": "surprise"}]
spark.createDataFrame(bad_data).write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/bad_data")

In [0]:
df_rescue = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_RESCUE)
    .option("cloudFiles.schemaEvolutionMode", "rescue")  # Rescue mode!
    .schema(partial_schema)  # Partial schema
    .load(STREAM_SOURCE_PATH)
)

**Start stream:**

In [0]:
query_rescue = (df_rescue.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_RESCUE)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE_RESCUE)
)
display(spark.table(TARGET_TABLE_RESCUE))

**Schema with `_rescued_data` column:**

In [0]:
spark.table(TARGET_TABLE_RESCUE).printSchema()

**Data with rescued columns:**

In [0]:
display(
    spark.table(TARGET_TABLE_RESCUE)
    .limit(5)
)

### Demo: badRecordsPath

Załadujemy celowo uszkodzone dane (nieprawidłowa data) i pokażemy jak `badRecordsPath` zapisuje odrzucone rekordy do katalogu jako JSON z kontekstem błędu (path, record, reason).

In [0]:
TABLE_ERRORS = f"{BRONZE_SCHEMA}.customers_with_validation"

**Creating table with `_corrupt_record`:**

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE_ERRORS}")

spark.sql(f"""
CREATE TABLE {TABLE_ERRORS} (
  customer_id STRING,
  first_name STRING,
  last_name STRING,
  email STRING,
  phone STRING,
  city STRING,
  state STRING,
  country STRING,
  registration_date DATE,
  customer_segment STRING,
  _corrupt_record STRING,
  _ingestion_timestamp TIMESTAMP
) USING DELTA
""")

**Loading with error handling:**

In [0]:
# Create a CSV with a bad row
bad_csv_data = [
    (999, "Eve", "2023-01-03"),
    (888, "Frank", "NOT_A_DATE") # This will fail date parsing
]
spark.createDataFrame(bad_csv_data, ["customer_id", "first_name", "registration_date"]).write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/bad_csv")

In [0]:
df_with_errors = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("badRecordsPath", BAD_RECORDS_PATH)
    .schema("""
        customer_id STRING,
        first_name STRING,
        registration_date DATE,
        _corrupt_record STRING
    """)
    .load(f"{BATCH_SOURCE_PATH}/bad_csv")
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)
display(df_with_errors)

**Bad records statistics:**

In [0]:
files = [f.path for f in dbutils.fs.ls(BAD_RECORDS_PATH)]
files = [f + "*" for f in files]
print(files)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

bad_record_schema = StructType([
    StructField("path", StringType(), True),
    StructField("record", StringType(), True),
    StructField("reason", StringType(), True)
])

df_bad_records = (
    spark.read
    .format("json")
    .schema(bad_record_schema)
    .load(files)
)
display(df_bad_records)

## Summary

| Topic | Key Concept | Key Terms |
|---|---|---|
| **COPY INTO** | Idempotent batch loading, file tracking | `COPY INTO`, `FILEFORMAT`, `mergeSchema` |
| **Auto Loader** | `cloudFiles` streaming ingestion | `cloudFiles.format`, `schemaLocation`, `schemaEvolutionMode` |
| **Trigger Modes** | `availableNow=True`, `processingTime` | Batch-like vs continuous |
| **pathGlobFilter** | Filter files by pattern in mixed directories | `pathGlobFilter`, `*.json`, `*.csv` |
| **Schema Evolution** | `rescue`, `addNewColumns`, `failOnNewColumns` | `_rescued_data` column |
| **Stream-Static Join** | Enrich stream with dimension table | Append mode, static reload |
| **Watermarking** | State cleanup in aggregations | `withWatermark()`, late data handling |
| **cleanSource** | Automatic file archival after processing | `cleanSource=MOVE`, `moveDestination`, `retentionDuration` |
| **Error Handling** | `PERMISSIVE`, `DROPMALFORMED`, `FAILFAST` | `badRecordsPath`, `_corrupt_record` |

In [0]:
# List of created tables
created_tables = [
    "customer_cts",
    "customer_cts_enriched",
    "customers_batch",
    "orders_autoloader",
    f"orders_autoloader_continuous",
    "orders_rescued",
    "customers_with_validation",
    "orders_archive_demo",
]

## Cleanup

Remove demo tables, checkpoints, and temporary data created during this module.

---

In [0]:
results = []
for table in created_tables:
    full_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table}"
    try:
        if spark.catalog.tableExists(full_table):
            count = spark.table(full_table).count()
            results.append((table, "EXISTS", str(count)))
        else:
            results.append((table, "NOT FOUND", "-"))
    except Exception as e:
        results.append((table, "ERROR", str(e)[:30]))

display(spark.createDataFrame(results, ["Table", "Status", "Records"]))

In [0]:
# Cleanup flag
CLEANUP_ENABLED = False

**Execute cleanup (if enabled):**

In [0]:
if CLEANUP_ENABLED:
    results = []
    for table in created_tables:
        full_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table}"
        try:
            spark.sql(f"DROP TABLE IF EXISTS {full_table}")
            results.append((table, "DROPPED"))
        except Exception as e:
            results.append((table, f"ERROR: {str(e)[:30]}"))
    
    # Cleanup checkpoints
    try:
        dbutils.fs.rm(CHECKPOINT_BASE_PATH, True)
        results.append(("checkpoints", "REMOVED"))
    except:
        results.append(("checkpoints", "NOT FOUND"))
    
    display(spark.createDataFrame(results, ["Resource", "Status"]))
else:
    display(spark.createDataFrame([
        ("CLEANUP_ENABLED", "False"),
        ("Action", "Change to True to delete resources")
    ], ["Setting", "Value"]))

---

← [01 — Medallion Architecture](01_medallion_architecture_demo.ipynb) | **[ README](../../../README.md)** | [03 — Lakeflow Pipelines](03_lakeflow_pipelines_demo.ipynb) →
